### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_CHWOZDVKHFOQ


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250318_squid_cal520"
recname = "rec11"
savedir = "/bucket/ReiterU/OLdev/analysis/20250602_OGL_IGL_clustering_2/20250318_rec11"

# templatefile = os.path.join(rootdir, "2P", recname, "run1/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
# roifile = os.path.join(rootdir, "2P", recname, "ROI_yutaka_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_yutaka/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi2

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719500,) (10949,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
print(F.shape) # F is sampled at 1 kH

(201, 719500)


### Compute ETA

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

### Save as HDF5

In [10]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)

### Plot cells

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

good_neurons = []
for nid in np.arange(0, F.shape[0]):
    q0, v0, n0 = compute_qi2(E, nid, 0)
    q1, v1, n1 = compute_qi2(E, nid, 1)
    q2, v2, n2 = compute_qi2(E, nid, 2)

    cond1 = (q0 > 0.25) and (v0 < 2.0)
    cond2 = (q1 > 0.25) and (v1 < 2.0)
    cond3 = (q2 > 0.25) and (v2 < 2.0)
    cond4 = max(v0, v1, v2) < 2.0
    cond5 = max(n0, n1, n2) < 0.3
    
    if (cond1 or cond2 or cond3) and cond4 and cond5:
    # if not((cond1 or cond2 or cond3) and cond4 and cond5):
        print("Neuron:", nid)
        print(f"q0={q0:.3f}, q1={q1:.3f}, q2={q2:.3f}")
        print(f"v0={v0:.3f}, v1={v1:.3f}, v2={v2:.3f}")
        print(f"n0={n0:.3f}, n1={n1:.3f}, n2={n2:.3f}")
        good_neurons.append(nid)
        plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=None)

In [12]:
print("All neurons=", F.shape[0])
print("good neurons=", len(good_neurons))
print("Percentage", f"{len(good_neurons)/F.shape[0]*100:.3f}%")
print(good_neurons)

All neurons= 201
good neurons= 124
Percentage 61.692%
[np.int64(1), np.int64(2), np.int64(3), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(23), np.int64(24), np.int64(30), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(37), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(57), np.int64(59), np.int64(60), np.int64(62), np.int64(66), np.int64(72), np.int64(80), np.int64(81), np.int64(82), np.int64(83), np.int64(84), np.int64(85), np.int64(88), np.int64(89), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(95), np.int64(96), np.int64(99), np.int64(101), np.int64(102), np.int64(103), np.int64(106), np.int64(109), np.int64(111), np.int64(112), np.int64(113), np.int64(114), np.int64(118), np.int64(119), np.int64(120), np.int64(121), 

### Copy data to bucket

In [13]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250602_OGL_IGL_clustering_2/20250318_rec11"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_CHWOZDVKHFOQ/* /bucket/ReiterU/OLdev/analysis/20250602_OGL_IGL_clustering_2/20250318_rec11"


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_CHWOZDVKHFOQ/* /bucket/ReiterU/OLdev/analysis/20250602_OGL_IGL_clustering_2/20250318_rec11"', returncode=0)